# Analyse optimisation runs

In [ ]:
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import argparse
import re

from pathlib import Path
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colorbar import Colorbar


import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_slice,
    plot_param_importances,
)

# Add parent directory to path
try:
    # For scripts
    sys.path.insert(0, str(Path(__file__).parent.parent.parent))
except NameError:
    # For Jupyter notebooks - add the known project root
    sys.path.insert(0, str(Path.cwd().parent.parent))

from ml_events_utils import stylesheet_default
from ml_events_utils import color_gio as color_dict
from ml_events_utils import color_deep as color_dict

import matplotlib.pyplot as plt
import numpy as np

from run_hyperparam_scan_optuna import create_study_summary_figure, apply_log_scale_to_y_axes

In [2]:
def replace(text: str) -> str:
    """
    Replace words for better plot labels.

    Note that "_" should be the last replacement to avoid interfering with other patterns.
    """
    replacements = {
        r"paper_momentum": r"$Z$+jet",
        r"_": " ",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return text

## Plotting of 5 optimization parameters

In [3]:
def generate_optimization_plots(study: optuna.Study,
                                valid_study: optuna.Study,
                                pdf_path: Path,
                                study_name: str,
                                storage: str) -> Path:
    """Generate and save optimization analysis plots to a multipage PDF.

    Uses matplotlib-based Optuna visualization functions for direct integration.

    Args:
        study: Loaded Optuna study object
        valid_study: Study object containing only valid trials
        pdf_path: Path to save the PDF
        study_name: Name of the study
        storage: Storage URL/path

    Returns:
        Path to generated PDF file
    """
    plt.style.use(stylesheet_default)

    # Save all figures to multipage PDF
    with PdfPages(str(pdf_path)) as pdf:
        d = pdf.infodict()
        d["Title"] = f"Optuna Study Analysis: {study_name}"
        d["Author"] = "Optuna"
        d["Subject"] = "Hyperparameter Optimization Analysis"
        d["Keywords"] = "Optuna, Hyperparameter Optimization"
        d["CreationDate"] = datetime.now()

        target_name="Validation Loss"

        # Page 1: Study summary
        fig_summary = create_study_summary_figure(study, study_name, storage, target_name=target_name)
        pdf.savefig(fig_summary, bbox_inches="tight")
        plt.close(fig_summary)

        # Page 2: Optimization history
        try:
            ax = plot_optimization_history(study, target_name=target_name)
            fig_history = plt.gcf()  # Get current figure
            apply_log_scale_to_y_axes(fig_history)  # Apply log scale to objective values
            ax.set_ylim(ymax=1e-3)  # Set upper limit for better visualization of differences
            ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences
            ax.set_xlim(xmin=-10)  # Start x-axis at 0 for better visualization of early trials

            ylabel = ax.get_ylabel()
            ax.set_ylabel(f"{ylabel.lower()}")
            xlabel = ax.get_xlabel()
            ax.set_xlabel(f"{xlabel.lower()}")

            legend = ax.get_legend().get_texts()
            legend_labels = [text.get_text().lower() for text in legend]
            print(f"Legend texts: {legend_labels}")
            ax.legend(labels=legend_labels, loc="upper right")

            ax.set_title("")  # Clear default title set by plot_optimization_history

            pdf.savefig(fig_history, bbox_inches="tight")
            plt.close(fig_history)
        except Exception as e:
            print(f"Warning: Could not generate optimization history plot: {e}")

        # Page 3: Slice plot (parameter importance via slices)
        try:
            axs = plot_slice(study, target_name=target_name)
            fig = plt.gcf()  # Get current figure (may have multiple subplots)

            for ax in axs:
                ax.set_yscale("log")  # Set log scale for y-axis of each subplot
                ax.set_ylim(ymax=1e-3)  # Set upper limit for better visualization of differences
                ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences

            axs[0].set_xlabel("batch size")
            axs[1].set_xlabel("learning rate")
            labels = [replace(label.get_text()) for label in axs[2].get_xticklabels()]
            axs[2].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')

            labels = [replace(label.get_text()) for label in axs[4].get_xticklabels()]
            axs[4].set_xticklabels(labels)

            # fig.set_title("")  # Clear default title set by plot_slice
            ylabel = axs[0].get_ylabel()
            axs[0].set_ylabel(f"{ylabel.lower()}")
            fig.suptitle("")  # Clear any default suptitle set by plot_slice
            # apply_log_scale_to_y_axes(fig)  # Apply log scale to objective values

            label = fig.axes[-1].get_ylabel()
            fig.axes[-1].set_ylabel(f"{label.lower()}")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            print(f"Warning: Could not generate slice plot: {e}")

        # Page 4: Parameter importance ranking (only if enough trials)
        if len(study.trials) >= 2:
            try:
                ax = plot_param_importances(study, target_name=target_name)
                fig_importance = plt.gcf()  # Get current figure
                # Note: importance plot usually has categorical x-axis, log scale not needed
                ax.set_xlabel(ax.get_xlabel().lower())
                ax.set_ylabel(ax.get_ylabel().lower())

                legend = ax.get_legend().get_texts()
                legend_labels = [text.get_text().lower() for text in legend]
                ax.legend(labels=legend_labels)

                labels = [replace(label.get_text()) for label in ax.get_yticklabels()]
                ax.set_yticklabels(labels)

                ax.set_title("")  # Clear default title set by plot_param_importances
                fig_importance.suptitle("")
                pdf.savefig(fig_importance, bbox_inches="tight")
                plt.close(fig_importance)
            except Exception as e:
                print(f"Warning: Could not generate parameter importance plot: {e}")

            try:
                ax = plot_param_importances(valid_study, target_name=target_name)
                fig_importance = plt.gcf()  # Get current figure
                # Note: importance plot usually has categorical x-axis, log scale not needed
                ax.set_xlabel(ax.get_xlabel().lower())
                ax.set_ylabel(ax.get_ylabel().lower())

                legend = ax.get_legend().get_texts()
                legend_labels = [text.get_text().lower() for text in legend]
                ax.legend(labels=legend_labels)

                labels = [replace(label.get_text()) for label in ax.get_yticklabels()]
                ax.set_yticklabels(labels)

                ax.set_title("")  # Clear default title set by plot_param_importances
                fig_importance.suptitle("")
                pdf.savefig(fig_importance, bbox_inches="tight")
                plt.close(fig_importance)
            except Exception as e:
                print(f"Warning: Could not generate parameter importance plot: {e}")

    return 0

### 1. large run Autoencoder

In [4]:
arguments = {
    "settings_file": "run_settings_scan.yaml",
    "study_name": "pol_scan",
    "storage": "sqlite:///../optimisation_runs/BatchNorm_large_Autoencoder/pol_scan.db",
    "output_root": "scan_runs",
    "plot_only": True
}

arg = argparse.Namespace(**arguments)

print("Arguments:")
for key, value in vars(arg).items():
    print(f"  {key}: {value}")

Arguments:
  settings_file: run_settings_scan.yaml
  study_name: pol_scan
  storage: sqlite:///../optimisation_runs/BatchNorm_large_1/pol_scan.db
  output_root: scan_runs
  plot_only: True


In [5]:
# Handle plot-only mode: load existing study and generate plots
try:
    study = optuna.load_study(study_name=arg.study_name, storage=arg.storage)
except Exception as e:
    print(f"Error loading study '{arg.study_name}' from storage '{arg.storage}': {e}")
    raise e

# Print study summary to console
best = study.best_trial
print(f"Study: {arg.study_name}")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {best.number}")
print(f"Best objective value: {best.value}")
print(f"Best parameters: {best.params}\n")

# Generate plots
print("Loading valid trials...")
valid_trials = []
for i, trial in enumerate(study.trials):
    # print(f"Trial {trial.number}: Value={trial.value}, Params={trial.params}, State={trial.state}")
    if trial.value == None:
        print(f"Warning: Trial {trial.number} has a None value.")
    elif trial.value > 1e15:
        print(f"Warning: Trial {trial.number} has an unusually high value of {trial.value}")
    else:
        valid_trials.append(trial)

valid__trial_study = optuna.create_study(study_name=f"{study.study_name}_valid", storage=study._storage, load_if_exists=True)
# .study.Study(study_name=study.study_name, storage=study.storage)
for valid_trial in valid_trials:
    valid__trial_study.add_trial(valid_trial)






Study: pol_scan
Total trials: 961
Best trial number: 501
Best objective value: 0.00022768646150687208
Best parameters: {'batch_size': 2048, 'learning_rate': 0.0031959142469131104, 'optimizer': 'AdamW', 'patience': 35, 'penalties': 'cross_section'}

Loading valid trials...


[I 2026-05-06 11:53:38,711] Using an existing study with name 'pol_scan_valid' instead of creating a new one.


In [6]:
print("Generating optimization analysis plots...")
try:
    pdf_path = Path.cwd() / Path(f"optimization_analysis_{arg.study_name}_large.pdf")
    generate_optimization_plots(study, valid__trial_study, pdf_path, arg.study_name, arg.storage)
    print(f"Plots saved to: {pdf_path}")
except Exception as e:
    print(f"Error generating plots: {e}")

Generating optimization analysis plots...


/tmp/ipykernel_105761/2470957773.py:40: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study, target_name=target_name)


Legend texts: ['validation loss', 'best value']


/tmp/ipykernel_105761/2470957773.py:66: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  axs = plot_slice(study, target_name=target_name)
[W 2026-05-06 11:55:40,734] Trial 10 is omitted in visualization because its objective value is inf or nan.
/tmp/ipykernel_105761/2470957773.py:77: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[2].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')
/tmp/ipykernel_105761/2470957773.py:80: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[4].set_xticklabels(labels)
/tmp/ipykernel_105761/2470957773.py:98: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interfac

Plots saved to: /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/final_results/plots/optimization_analysis_pol_scan_large.pdf


In [7]:
print("Best parameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best parameters found:
  batch_size: 2048
  learning_rate: 0.0031959142469131104
  optimizer: AdamW
  patience: 35
  penalties: cross_section


- `batch_size` = 2048 [Most important!]
- `learning_rate` ~ 0.0032
- `optimizer` = AdamW
- `patience` = 35 (or just $>= 20$)
- `penalty` = doesn't really matter

### Standardised run FFNN_paper_4extraLayers

In [8]:
arguments = {
    "settings_file": "run_settings_scan.yaml",
    "study_name": "pol_scan",
    "storage": "sqlite:///../optimisation_runs/standardised_FFNN_paper_4extraLayers/pol_scan.db",
    "output_root": "scan_runs",
    "plot_only": True
}

arg = argparse.Namespace(**arguments)

print("Arguments:")
for key, value in vars(arg).items():
    print(f"  {key}: {value}")

Arguments:
  settings_file: run_settings_scan.yaml
  study_name: pol_scan
  storage: sqlite:///../optimisation_runs/standardised/pol_scan.db
  output_root: scan_runs
  plot_only: True


In [9]:
# Handle plot-only mode: load existing study and generate plots
try:
    study = optuna.load_study(study_name=arg.study_name, storage=arg.storage)
except Exception as e:
    print(f"Error loading study '{arg.study_name}' from storage '{arg.storage}': {e}")
    raise e

# Print study summary to console
best = study.best_trial
print(f"Study: {arg.study_name}")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {best.number}")
print(f"Best objective value: {best.value}")
print(f"Best parameters: {best.params}\n")

# Generate plots
print("Loading valid trials...")
valid_trials = []
for i, trial in enumerate(study.trials):
    # print(f"Trial {trial.number}: Value={trial.value}, Params={trial.params}, State={trial.state}")
    if trial.value == None:
        print(f"Warning: Trial {trial.number} has a None value.")
    elif trial.value > 1e15:
        print(f"Warning: Trial {trial.number} has an unusually high value of {trial.value}")
    else:
        valid_trials.append(trial)

valid__trial_study = optuna.create_study(study_name=f"{study.study_name}_valid", storage=study._storage, load_if_exists=True)
# .study.Study(study_name=study.study_name, storage=study.storage)
for valid_trial in valid_trials:
    valid__trial_study.add_trial(valid_trial)


Study: pol_scan
Total trials: 1034
Best trial number: 600
Best objective value: 0.00020219811374915645
Best parameters: {'batch_size': 256, 'learning_rate': 3.1841878135783404e-05, 'optimizer': 'paper_momentum', 'patience': 40, 'penalties': None}

Loading valid trials...


[I 2026-05-06 11:56:07,487] Using an existing study with name 'pol_scan_valid' instead of creating a new one.


In [10]:
print("Generating optimization analysis plots...")
try:
    pdf_path = Path.cwd() / Path(f"optimization_analysis_{arg.study_name}_standardised.pdf")
    generate_optimization_plots(study, valid__trial_study, pdf_path, arg.study_name, arg.storage)
    print(f"Plots saved to: {pdf_path}")
except Exception as e:
    print(f"Error generating plots: {e}")

Generating optimization analysis plots...


/tmp/ipykernel_105761/2470957773.py:40: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study, target_name=target_name)


Legend texts: ['validation loss', 'best value']


/tmp/ipykernel_105761/2470957773.py:66: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  axs = plot_slice(study, target_name=target_name)
[W 2026-05-06 11:58:01,014] Trial 2 is omitted in visualization because its objective value is inf or nan.
/tmp/ipykernel_105761/2470957773.py:77: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[2].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')
/tmp/ipykernel_105761/2470957773.py:80: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[4].set_xticklabels(labels)
/tmp/ipykernel_105761/2470957773.py:98: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface

Plots saved to: /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/final_results/plots/optimization_analysis_pol_scan_standardised.pdf


Results of the first large fit

In [11]:
print("Best parameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best parameters found:
  batch_size: 256
  learning_rate: 3.1841878135783404e-05
  optimizer: paper_momentum
  patience: 40
  penalties: None


- `batch_size` = 256 (as small as possible?)
- `learning_rate` ~ 0.000032 (as small as possible?)  [Most important!]
- `optimizer` = paper_momentum (AdamW similar good)
- `patience` = 40 (or just $>= 20$)
- `penalty` = doesn't really matter

## Plotting of 2 optimization parameters

In [22]:
def generate_optimization_plots_2(study: optuna.Study,
                                  valid_study: optuna.Study,
                                  pdf_path: Path,
                                  study_name: str,
                                  storage: str) -> Path:
    """Generate and save optimization analysis plots to a multipage PDF.

    Uses matplotlib-based Optuna visualization functions for direct integration.

    Args:
        study: Loaded Optuna study object
        valid_study: Study object containing only valid trials
        pdf_path: Path to save the PDF
        study_name: Name of the study
        storage: Storage URL/path

    Returns:
        Path to generated PDF file
    """
    plt.style.use(stylesheet_default)

    # Save all figures to multipage PDF
    with PdfPages(str(pdf_path)) as pdf:
        d = pdf.infodict()
        d["Title"] = f"Optuna Study Analysis: {study_name}"
        d["Author"] = "Optuna"
        d["Subject"] = "Hyperparameter Optimization Analysis"
        d["Keywords"] = "Optuna, Hyperparameter Optimization"
        d["CreationDate"] = datetime.now()

        target_name="Validation Loss"

        # Page 1: Study summary
        fig_summary = create_study_summary_figure(study, study_name, storage, target_name=target_name)
        pdf.savefig(fig_summary, bbox_inches="tight")
        plt.close(fig_summary)

        # Page 2: Optimization history
        try:
            ax = plot_optimization_history(study, target_name=target_name)
            fig_history = plt.gcf()  # Get current figure
            apply_log_scale_to_y_axes(fig_history)  # Apply log scale to objective values
            # ax.set_ylim(ymax=4.5e-4)  # Set upper limit for better visualization of differences
            # ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences

            ylabel = ax.get_ylabel()
            ax.set_ylabel(f"{ylabel.lower()}")
            xlabel = ax.get_xlabel()
            ax.set_xlabel(f"{xlabel.lower()}")

            legend = ax.get_legend().get_texts()
            legend_labels = [text.get_text().lower() for text in legend]
            print(f"Legend texts: {legend_labels}")
            ax.legend(labels=legend_labels, loc="upper right")

            ax.set_title("")  # Clear default title set by plot_optimization_history

            pdf.savefig(fig_history, bbox_inches="tight")
            plt.close(fig_history)
        except Exception as e:
            print(f"Warning: Could not generate optimization history plot: {e}")

        # Page 3: Slice plot (parameter importance via slices)
        try:
            axs = plot_slice(study, target_name=target_name)
            fig = plt.gcf()  # Get current figure (may have multiple subplots)

            for ax in axs:
                ax.set_yscale("log")  # Set log scale for y-axis of each subplot
                # ax.set_ylim(ymax=1e-3)  # Set upper limit for better visualization of differences
                # ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences

            for ax in axs:
                ax.set_xlabel(replace(ax.get_xlabel()).lower())

            labels = [replace(label.get_text()) for label in axs[1].get_xticklabels()]
            axs[1].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')

            # fig.set_title("")  # Clear default title set by plot_slice
            ylabel = axs[0].get_ylabel()
            axs[0].set_ylabel(f"{ylabel.lower()}")
            fig.suptitle("")  # Clear any default suptitle set by plot_slice
            # apply_log_scale_to_y_axes(fig)  # Apply log scale to objective values

            label = fig.axes[-1].get_ylabel()
            fig.axes[-1].set_ylabel(f"{label.lower()}")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            print(f"Warning: Could not generate slice plot: {e}")

        # Page 4: Parameter importance ranking (only if enough trials)
        if len(study.trials) >= 2:
            try:
                ax = plot_param_importances(study, target_name=target_name)
                fig_importance = plt.gcf()  # Get current figure
                # Note: importance plot usually has categorical x-axis, log scale not needed
                ax.set_xlabel(ax.get_xlabel().lower())
                ax.set_ylabel(ax.get_ylabel().lower())

                legend = ax.get_legend().get_texts()
                legend_labels = [text.get_text().lower() for text in legend]
                ax.legend(labels=legend_labels)

                labels = [replace(label.get_text()) for label in ax.get_yticklabels()]
                ax.set_yticklabels(labels)

                ax.set_title("")  # Clear default title set by plot_param_importances
                fig_importance.suptitle("")
                pdf.savefig(fig_importance, bbox_inches="tight")
                plt.close(fig_importance)
            except Exception as e:
                print(f"Warning: Could not generate parameter importance plot: {e}")

            try:
                ax = plot_param_importances(valid_study, target_name=target_name)
                fig_importance = plt.gcf()  # Get current figure
                # Note: importance plot usually has categorical x-axis, log scale not needed
                ax.set_xlabel(ax.get_xlabel().lower())
                ax.set_ylabel(ax.get_ylabel().lower())

                legend = ax.get_legend().get_texts()
                legend_labels = [text.get_text().lower() for text in legend]
                ax.legend(labels=legend_labels)

                labels = [replace(label.get_text()) for label in ax.get_yticklabels()]
                ax.set_yticklabels(labels)

                ax.set_title("")  # Clear default title set by plot_param_importances
                fig_importance.suptitle("")
                pdf.savefig(fig_importance, bbox_inches="tight")
                plt.close(fig_importance)
            except Exception as e:
                print(f"Warning: Could not generate parameter importance plot: {e}")

    return 0

### BatchNorm_2 FFNN_paper_4extraLayers

In [23]:
arguments = {
    "settings_file": "run_settings_scan.yaml",
    "study_name": "pol_scan",
    "storage": "sqlite:///../optimisation_runs/BatchNorm_2_FFNN_paper_4extraLayers/pol_scan.db",
    "output_root": "scan_runs",
    "plot_only": True
}

arg = argparse.Namespace(**arguments)

print("Arguments:")
for key, value in vars(arg).items():
    print(f"  {key}: {value}")

Arguments:
  settings_file: run_settings_scan.yaml
  study_name: pol_scan
  storage: sqlite:///../optimisation_runs/BatchNorm_2/pol_scan.db
  output_root: scan_runs
  plot_only: True


In [24]:
# Handle plot-only mode: load existing study and generate plots
try:
    study = optuna.load_study(study_name=arg.study_name, storage=arg.storage)
except Exception as e:
    print(f"Error loading study '{arg.study_name}' from storage '{arg.storage}': {e}")
    raise e

# Print study summary to console
best = study.best_trial
print(f"Study: {arg.study_name}")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {best.number}")
print(f"Best objective value: {best.value}")
print(f"Best parameters: {best.params}\n")

# Generate plots
print("Loading valid trials...")
valid_trials = []
for i, trial in enumerate(study.trials):
    # print(f"Trial {trial.number}: Value={trial.value}, Params={trial.params}, State={trial.state}")
    if trial.value == None:
        print(f"Warning: Trial {trial.number} has a None value.")
    elif trial.value > 1e15:
        print(f"Warning: Trial {trial.number} has an unusually high value of {trial.value}")
    else:
        valid_trials.append(trial)

valid__trial_study = optuna.create_study(study_name=f"{study.study_name}_valid", storage=study._storage, load_if_exists=True)
# .study.Study(study_name=study.study_name, storage=study.storage)
for valid_trial in valid_trials:
    valid__trial_study.add_trial(valid_trial)


Study: pol_scan


[I 2026-05-06 12:00:32,927] Using an existing study with name 'pol_scan_valid' instead of creating a new one.


Total trials: 173
Best trial number: 128
Best objective value: 0.0003712478941952389
Best parameters: {'learning_rate': 0.0002830145196511958, 'penalties': 'cross_section'}

Loading valid trials...


In [25]:
print("Generating optimization analysis plots...")
try:
    pdf_path = Path.cwd() / Path(f"optimization_analysis_{arg.study_name}_BatchNorm_2.pdf")
    generate_optimization_plots_2(study, valid__trial_study, pdf_path, arg.study_name, arg.storage)
    print(f"Plots saved to: {pdf_path}")
except Exception as e:
    print(f"Error generating plots: {e}")

Generating optimization analysis plots...


/tmp/ipykernel_105761/1764180483.py:40: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study, target_name=target_name)


Legend texts: ['validation loss', 'best value']


/tmp/ipykernel_105761/1764180483.py:65: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  axs = plot_slice(study, target_name=target_name)
/tmp/ipykernel_105761/1764180483.py:77: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[1].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')
/tmp/ipykernel_105761/1764180483.py:95: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_param_importances(study, target_name=target_name)
/tmp/ipykernel_105761/1764180483.py:116: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  

Plots saved to: /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/final_results/plots/optimization_analysis_pol_scan_BatchNorm_2.pdf


Results of the first large fit

In [26]:
print("Best parameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best parameters found:
  learning_rate: 0.0002830145196511958
  penalties: cross_section


- `learning_rate` ~ 0.00028 [Most important!]
- `penalty` = doesn't really matter

### BatchNorm NLOPS Autoencoder

In [27]:
arguments = {
    "settings_file": "run_settings_scan.yaml",
    "study_name": "pol_scan",
    "storage": "sqlite:///../optimisation_runs/BatchNorm_NLOPS_Autoencoder/pol_scan.db",
    "output_root": "scan_runs",
    "plot_only": True
}

arg = argparse.Namespace(**arguments)

print("Arguments:")
for key, value in vars(arg).items():
    print(f"  {key}: {value}")

Arguments:
  settings_file: run_settings_scan.yaml
  study_name: pol_scan
  storage: sqlite:///../optimisation_runs/BatchNorm_NLOPS/pol_scan.db
  output_root: scan_runs
  plot_only: True


In [28]:
# Handle plot-only mode: load existing study and generate plots
try:
    study = optuna.load_study(study_name=arg.study_name, storage=arg.storage)
except Exception as e:
    print(f"Error loading study '{arg.study_name}' from storage '{arg.storage}': {e}")
    raise e

# Print study summary to console
best = study.best_trial
print(f"Study: {arg.study_name}")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {best.number}")
print(f"Best objective value: {best.value}")
print(f"Best parameters: {best.params}\n")

# Generate plots
print("Loading valid trials...")
valid_trials = []
for i, trial in enumerate(study.trials):
    # print(f"Trial {trial.number}: Value={trial.value}, Params={trial.params}, State={trial.state}")
    if trial.value == None:
        print(f"Warning: Trial {trial.number} has a None value.")
    elif trial.value > 1e15:
        print(f"Warning: Trial {trial.number} has an unusually high value of {trial.value}")
    else:
        valid_trials.append(trial)

valid__trial_study = optuna.create_study(study_name=f"{study.study_name}_valid", storage=study._storage, load_if_exists=True)
# .study.Study(study_name=study.study_name, storage=study.storage)
for valid_trial in valid_trials:
    valid__trial_study.add_trial(valid_trial)


Study: pol_scan


[I 2026-05-06 12:01:03,801] Using an existing study with name 'pol_scan_valid' instead of creating a new one.


Total trials: 175
Best trial number: 85
Best objective value: 0.0011154285628232174
Best parameters: {'learning_rate': 0.0009052673891404944, 'penalties': 'cross_section'}

Loading valid trials...


In [29]:
print("Generating optimization analysis plots...")
try:
    pdf_path = Path.cwd() / Path(f"optimization_analysis_{arg.study_name}_BatchNorm_NLOPS.pdf")
    generate_optimization_plots_2(study, valid__trial_study, pdf_path, arg.study_name, arg.storage)
    print(f"Plots saved to: {pdf_path}")
except Exception as e:
    print(f"Error generating plots: {e}")

Generating optimization analysis plots...


/tmp/ipykernel_105761/1764180483.py:40: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study, target_name=target_name)


Legend texts: ['validation loss', 'best value']


/tmp/ipykernel_105761/1764180483.py:65: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  axs = plot_slice(study, target_name=target_name)
/tmp/ipykernel_105761/1764180483.py:77: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[1].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')
/tmp/ipykernel_105761/1764180483.py:95: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_param_importances(study, target_name=target_name)
/tmp/ipykernel_105761/1764180483.py:116: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  

Plots saved to: /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/final_results/plots/optimization_analysis_pol_scan_BatchNorm_NLOPS.pdf


Results of the first large fit

In [30]:
print("Best parameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best parameters found:
  learning_rate: 0.0009052673891404944
  penalties: cross_section


- `learning_rate` ~ 0.00091 [Most important!]
- `penalty` = doesn't really matter

# Summary

## Autoencoder (with BatchNorm)
- 1. large run (NLO):
    - `batch_size` = 2048 [Most important!]
    - `learning_rate` ~ 0.0032
    - `optimizer` = AdamW
    - `patience` = 35 (or just $>= 20$)
- 2. NLOPS:
    - `learning_rate` ~ 0.00091 [Most important!]

## FFNN_paper_4extra_layers
- Standardised (NLO):
    - `batch_size` = 256 (as small as possible?)
    - `learning_rate` ~ 0.000032 (as small as possible?)  [Most important!]
    - `optimizer` = paper_momentum (AdamW similar good)
    - `patience` = 40 (or just $>= 20$)
- BatchNorm (NLO):
    - `learning_rate` ~ 0.00028 [Most important!]


# TODO:
- Have the following fixed for BatchNorm runs:
    - `batch_size` = 2048
    - `optimizer`  = AdamW
    - `patience`   = 35
- Vary the `learning_rate` $\in [5 \cdot 10^{-5}, 8 \cdot 10^{-3}]$:
    - for Autoencoder at NLO and NLOPS
    - for FFNN_paper_4extra_layers at NLO and NLOPS